# Data Structuring & Combining — PT

## Learning goals
By the end of this lesson, you should be able to:
- decide whether datasets should be **stacked** or **matched**;
- combine DataFrames vertically with `pd.concat()`;
- combine related DataFrames using `pd.merge()`;
- recognize `inner`, `left`, `right`, and `outer` merges;
- create summary tables with `pivot_table()`;
- recognize **wide** and **long** data;
- reshape wide data into long format with `melt()`.

> **What structure does the analysis need?**

## 1. Why combine or restructure data?
Real data is often distributed across several tables. A logistics company might have monthly delivery files plus separate driver information.

Sometimes we need to **add more observations**. Sometimes we need to **add information about the same observations**. These are different problems.

In [ ]:
import pandas as pd

january = pd.DataFrame({
    "delivery_id": [101, 102, 103],
    "driver_id": ["D01", "D02", "D03"],
    "zone": ["North", "South", "North"],
    "distance_km": [12, 8, 15]
})
february = pd.DataFrame({
    "delivery_id": [104, 105, 106],
    "driver_id": ["D02", "D04", "D01"],
    "zone": ["East", "South", "East"],
    "distance_km": [10, 18, 7]
})
drivers = pd.DataFrame({
    "driver_id": ["D01", "D02", "D03", "D04"],
    "driver_name": ["Ana", "Bruno", "Carla", "David"],
    "vehicle": ["Van", "Bike", "Van", "Truck"]
})

display(january)
display(february)
display(drivers)

## 2. `concat()` — stacking compatible datasets
January and February contain the **same variables** but different deliveries. We simply need to put February **under** January.

In [ ]:
deliveries = pd.concat([january, february], axis=0, ignore_index=True)
display(deliveries)

### Understanding `axis`
- `axis=0` → stack vertically → add rows.
- `axis=1` → place horizontally → add columns.

For monthly files with the same variables, `axis=0` is normally appropriate. `ignore_index=True` creates a new sequential row index.

### 💡 Quick Check 1 — Complete the code
Add March observations underneath `deliveries`.

In [ ]:
march = pd.DataFrame({
    "delivery_id": [107, 108],
    "driver_id": ["D03", "D04"],
    "zone": ["North", "South"],
    "distance_km": [11, 14]
})

all_deliveries = pd.________(
    [deliveries, ________],
    axis=_____,
    ignore_index=True
)
display(all_deliveries)

## 3. `merge()` — matching related information
The deliveries contain `driver_id`, while `drivers` contains names and vehicles. Both tables share the key `driver_id`, so we can **match related rows**.

In [ ]:
delivery_details = pd.merge(deliveries, drivers, on="driver_id")
display(delivery_details)

By default, `merge()` performs an **inner merge**: only keys found in both DataFrames are kept.

```text
concat → stack compatible tables
merge  → match related tables using a key
```

### 💡 Quick Check 2 — Complete the code
Combine `deliveries` and `drivers` using `driver_id`.

In [ ]:
combined = pd.________(
    deliveries,
    drivers,
    on=________
)
display(combined)

## 4. Choosing the merge type
Not every key necessarily appears in both DataFrames.

In [ ]:
routes = pd.DataFrame({
    "route_id": ["R1", "R2", "R3"],
    "zone": ["North", "South", "East"]
})
incidents = pd.DataFrame({
    "route_id": ["R2", "R3", "R4"],
    "incidents": [2, 1, 3]
})
display(routes)
display(incidents)

The `how` parameter determines which keys are kept.

| `how` | Keeps |
|---|---|
| `"inner"` | only keys present in both tables |
| `"left"` | all keys from the left table |
| `"right"` | all keys from the right table |
| `"outer"` | all keys from either table |

> **Which observations do I need to preserve?**

In [ ]:
print("INNER")
display(pd.merge(routes, incidents, on="route_id", how="inner"))

print("LEFT")
display(pd.merge(routes, incidents, on="route_id", how="left"))

print("RIGHT")
display(pd.merge(routes, incidents, on="route_id", how="right"))

print("OUTER")
display(pd.merge(routes, incidents, on="route_id", how="outer"))

### Different key names
If matching columns contain the same information but have different names, use `left_on` and `right_on`.

In [ ]:
zones = pd.DataFrame({
    "zone_code": ["North", "South", "East"],
    "manager": ["Laura", "Miguel", "Sara"]
})

deliveries_with_manager = pd.merge(
    deliveries, zones,
    left_on="zone", right_on="zone_code",
    how="left"
)
display(deliveries_with_manager)

### 💡 Quick Check 3 — Complete the code
Keep **all routes** and add incident information when it exists.

In [ ]:
routes_with_incidents = pd.merge(
    routes,
    incidents,
    on="route_id",
    how=________
)
display(routes_with_incidents)

## 5. Verify after combining data
A merge can run successfully and still produce an unexpected result. Check dimensions, missing values, keys and columns.

In [ ]:
print("Before merge:", deliveries.shape)
print("After merge:", delivery_details.shape)

print("\nMissing values:")
display(delivery_details.isna().sum())

print("\nColumns:")
print(delivery_details.columns.tolist())

### 💡 Quick Check 4 — Complete the code
Verify the shape and missing values of `routes_with_incidents`.

In [ ]:
print(routes_with_incidents._____)
display(routes_with_incidents._____()._____())

## 6. Structuring data for a question
Sometimes all the information is already in one DataFrame, but its layout does not make the question easy to answer.

Suppose the operations manager wants to compare total delivery distance by **zone** and **vehicle**.

In [ ]:
display(delivery_details)

A row-level table is useful for storing observations. A summary table may be more useful for answering a specific question.

## 7. `pivot_table()` — building a summary table
A pivot table reorganizes data and performs an aggregation.

- `index` → rows
- `columns` → columns
- `values` → measure
- `aggfunc` → summary operation

In [ ]:
distance_summary = delivery_details.pivot_table(
    index="zone",
    columns="vehicle",
    values="distance_km",
    aggfunc="sum"
)
display(distance_summary)

This answers: **How much total delivery distance was covered in each zone by each vehicle type?**

```text
Rows?      → zone
Columns?   → vehicle
Measure?   → distance_km
Summary?   → sum
```

### 💡 Quick Check 5 — Complete the code
Create a pivot table showing the **average** delivery distance for each driver and zone.

In [ ]:
average_distance = delivery_details.________(
    index=________,
    columns=________,
    values=________,
    aggfunc=________
)
display(average_distance)

## 8. Wide and long data
The same information can be organized in different shapes.

### Wide format
Different repeated measurements occupy different columns.

In [ ]:
monthly_distance = pd.DataFrame({
    "driver": ["Ana", "Bruno", "Carla"],
    "jan_km": [120, 95, 110],
    "feb_km": [135, 105, 100],
    "mar_km": [128, 115, 125]
})
display(monthly_distance)

### Long format
The month can become a variable of its own:

```text
driver   month   distance_km
Ana      jan     120
Ana      feb     135
Ana      mar     128
```

Long format is often convenient because each column represents a variable.

## 9. `melt()` — wide to long
`melt()` transforms repeated-measurement columns into rows.

- `id_vars` → identifier columns
- `value_vars` → columns to reshape
- `var_name` → new variable column
- `value_name` → new values column

In [ ]:
distance_long = pd.melt(
    monthly_distance,
    id_vars=["driver"],
    value_vars=["jan_km", "feb_km", "mar_km"],
    var_name="month",
    value_name="distance_km"
)
display(distance_long)

### 💡 Quick Check 6 — Complete the code
Transform the wide table into long format. Keep `warehouse` as identifier and create `quarter` and `orders` columns.

In [ ]:
quarterly_orders = pd.DataFrame({
    "warehouse": ["Madrid", "Barcelona"],
    "q1": [320, 280],
    "q2": [350, 310]
})

orders_long = pd.________(
    quarterly_orders,
    id_vars=[________],
    value_vars=[________, ________],
    var_name=________,
    value_name=________
)
display(orders_long)

## 10. Choosing the right transformation

- Compatible tables with the same variables? → `pd.concat(...)`
- Related tables sharing a key? → `pd.merge(...)`
- Need an aggregated cross-tabular summary? → `df.pivot_table(...)`
- Repeated measurements in columns need to become rows? → `pd.melt(...)`

> **Restructure data when a different shape makes the analysis easier.**

## 11. Final Challenge — Complete the transformation
Complete the code to stack monthly orders, add warehouse information, and summarize total units by city and product.

In [ ]:
jan_orders = pd.DataFrame({
    "order_id": [1, 2, 3],
    "warehouse_id": ["W1", "W2", "W1"],
    "product": ["A", "A", "B"],
    "units": [10, 8, 12]
})
feb_orders = pd.DataFrame({
    "order_id": [4, 5],
    "warehouse_id": ["W2", "W3"],
    "product": ["B", "A"],
    "units": [7, 9]
})
warehouses = pd.DataFrame({
    "warehouse_id": ["W1", "W2", "W3"],
    "city": ["Madrid", "Barcelona", "Valencia"]
})

orders = pd.________(
    [jan_orders, feb_orders],
    ignore_index=True
)

orders = pd.________(
    orders,
    warehouses,
    on=________,
    how=________
)

summary = orders.________(
    index=________,
    columns=________,
    values=________,
    aggfunc=________
)

display(summary)

## Summary
Today you learned to choose a transformation based on the structure of the problem:

**Stack → Match → Summarize → Reshape**

- `concat()` stacks compatible datasets;
- `merge()` matches related datasets using a key;
- `pivot_table()` creates aggregated summary tables;
- `melt()` transforms wide data into long format.

### Further exploration
Pandas also provides:
- horizontal concatenation with `axis=1`;
- `join()` for index-based combinations;
- `stack()` and `unstack()`;
- multi-level indexes;
- more advanced pivot-table options.

These tools can be introduced when a dataset or analysis requires them.